# Model Monitoring: Population Stability Index (PSI)

In this notebook, we calculate the Population Stability Index (PSI) to monitor our PD model.

The objective is to:

- Compare the distribution of model inputs and scores between the training dataset and a new dataset
- Measure whether the population has shifted
- Quantify that shift using PSI

Why this matters:

A model may perform well during development, but if the population changes over time, its predictions may become unreliable. PSI helps us detect such data drift.


In [1]:
import numpy as np
import pandas as pd


In [2]:
train_data = pd.read_csv('../data/processed/pd_model_train_woe.csv')
test_data  = pd.read_csv('../data/processed/pd_model_test_woe.csv')

In [3]:
train_data.shape
test_data.shape


(37500, 5)

In [4]:
train_data.columns.equals(test_data.columns)


True

In [5]:
train_data.head()


,age_woe,DPD_30_59_woe,DPD_60_89_woe,MonthlyIncome_woe,SeriousDlqin2yrs
0,0.290571,0.532507,0.289418,0.345527,0
1,-0.214610,0.532507,0.289418,0.198608,0
2,0.506623,-1.079482,0.289418,-0.401421,0
3,0.290571,0.532507,0.289418,-0.284439,0
4,-0.214610,0.532507,0.289418,0.198608,0


## Why We Are Calculating Score-Level PSI (Not Input Variable PSI)

In this notebook, we focus on calculating PSI on the model score (PD output) rather than on individual input variables.

While input variable drift measures whether features such as age, income, or delinquency have shifted over time, banks and risk teams are primarily concerned with model output drift.

The reason is:

The final credit decision is based on the model score (PD), not individual features.

Even small shifts across multiple inputs can compound into significant changes in predicted risk.

Score-level PSI directly shows whether the portfolio risk profile has changed.

Therefore, we calculate PSI on predicted PD values to monitor model stability in a way that reflects real business impact.

In [6]:
# Separate features and target

X_train = train_data.drop('SeriousDlqin2yrs', axis=1)
y_train = train_data['SeriousDlqin2yrs']

X_test = test_data.drop('SeriousDlqin2yrs', axis=1)
y_test = test_data['SeriousDlqin2yrs']


In [7]:
X_train.shape
X_test.shape


(37500, 4)

### Generating Predicted PD Values for Monitoring

To calculate Score-Level PSI, we need predicted PD values for both:

Development population (training data)

New population (test data)

Even though the model was built earlier, we rebuild it here to ensure:

The same trained model is used consistently

Predicted probabilities are generated directly within this monitoring notebook

The monitoring process is fully reproducible and self-contained

We will:

Train the logistic regression model on training data

Generate predicted probabilities for both train and test datasets

Use those probabilities to calculate PSI

In [8]:
from sklearn.linear_model import LogisticRegression

# Initialize logistic regression
log_reg = LogisticRegression(max_iter=1000)

# Fit model on training data only
log_reg.fit(X_train, y_train)

print("Model trained successfully.")


Model trained successfully.


In [9]:
# Generate predicted probabilities (PD)

train_data['PD'] = log_reg.predict_proba(X_train)[:, 1]
test_data['PD'] = log_reg.predict_proba(X_test)[:, 1]

print("Predicted PD values generated.")


Predicted PD values generated.


In [12]:
train_data['PD'].describe()


count    112500.000000
mean          0.066928
std           0.092218
min           0.011989
25%           0.026276
50%           0.039445
75%           0.061320
max           0.852608
Name: PD, dtype: float64

In [13]:
test_data['PD'].describe()

count    37500.000000
mean         0.066512
std          0.092226
min          0.011989
25%          0.025614
50%          0.039445
75%          0.061320
max          0.852608
Name: PD, dtype: float64

### Creating Fixed PD Buckets

To calculate PSI, we divide predicted PD values into fixed probability bands.

It is important to use fixed bins (not percentile-based bins), because PSI measures distribution shift relative to a fixed reference structure.

If percentile bins were used, both datasets would automatically have equal proportions, making PSI meaningless.


In [14]:
# Define fixed PD bins
bins = [0, 0.01, 0.03, 0.05, 0.10, 0.20, 1]

train_data['PD_Bucket'] = pd.cut(train_data['PD'], bins=bins)
test_data['PD_Bucket'] = pd.cut(test_data['PD'], bins=bins)


In [16]:
train_data['PD_Bucket'].value_counts(normalize=True).sort_index()


PD_Bucket
(0.0, 0.01]     0.000000
(0.01, 0.03]    0.298124
(0.03, 0.05]    0.329787
(0.05, 0.1]     0.226240
(0.1, 0.2]      0.093182
(0.2, 1.0]      0.052667
Name: proportion, dtype: float64

In [17]:
test_data['PD_Bucket'].value_counts(normalize=True).sort_index()

PD_Bucket
(0.0, 0.01]     0.000000
(0.01, 0.03]    0.304427
(0.03, 0.05]    0.325440
(0.05, 0.1]     0.226373
(0.1, 0.2]      0.091920
(0.2, 1.0]      0.051840
Name: proportion, dtype: float64

### Calculating Population Stability Index (PSI)

We now compute PSI by comparing the proportion of observations in each PD bucket between:

- Training data (reference population)
- Test data (new population)

PSI formula:

PSI = (P_new − P_ref) × ln(P_new / P_ref)

We sum the contribution across all buckets to get total PSI.


In [18]:
# Calculate distributions
train_dist = train_data['PD_Bucket'].value_counts(normalize=True).sort_index()
test_dist = test_data['PD_Bucket'].value_counts(normalize=True).sort_index()

# Combine into one dataframe
psi_df = pd.concat([train_dist, test_dist], axis=1)
psi_df.columns = ['Train_Proportion', 'Test_Proportion']

# Replace zeros to avoid log errors
psi_df = psi_df.replace(0, 0.0001)

# Calculate PSI contribution per bucket
psi_df['PSI'] = (
    (psi_df['Test_Proportion'] - psi_df['Train_Proportion']) *
    np.log(psi_df['Test_Proportion'] / psi_df['Train_Proportion'])
)

# Total PSI
total_psi = psi_df['PSI'].sum()

psi_df, total_psi


(              Train_Proportion  Test_Proportion           PSI
 PD_Bucket                                                    
 (0.0, 0.01]           0.000100         0.000100  0.000000e+00
 (0.01, 0.03]          0.298124         0.304427  1.318376e-04
 (0.03, 0.05]          0.329787         0.325440  5.767100e-05
 (0.05, 0.1]           0.226240         0.226373  7.855614e-08
 (0.1, 0.2]            0.093182         0.091920  1.721459e-05
 (0.2, 1.0]            0.052667         0.051840  1.307844e-05,
 0.0002198802102748509)

## PSI Result Interpretation

The total PSI between the development (train) and monitoring (test) datasets is:

PSI ≈ 0.00002

Interpretation:
- PSI < 0.1 → No significant population shift
- 0.1 ≤ PSI < 0.25 → Moderate shift
- PSI ≥ 0.25 → Major shift

Conclusion:
The population remains highly stable. No material data drift is observed.
The PD model can be considered stable from a population perspective.
